# Project: Video Game Market Dataset Preprocessing

- Author: Nurassyl Molshin
- Date: 2026-08-15

### Project goals and objectives

**Project goal** — to prepare a cleaned and properly preprocessed slice of historical data on games (2000-2013) that the "Secrets of Darkwood" team can use for an article about the development of the gaming industry in the early 21st century, with a focus on the RPG genre.

**Objectives:**
1. Load the data and take a first look at it.
2. Find and fix errors in the data: incorrect column names, wrong data types, missing values, explicit and implicit duplicates.
3. Filter the data by the 2000-2013 period, inclusive.
4. Categorize the games by user and critic scores (high / medium / low).
5. Identify the top 7 platforms by the number of games released during the period under review.

### Data description

The file `/datasets/new_games.csv` contains information about game sales and scores:

| Column | Description |
|---|---|
| Name | game title |
| Platform | platform name |
| Year of Release | year the game was released |
| Genre | game genre |
| NA sales | sales in North America, millions of copies |
| EU sales | sales in Europe, millions of copies |
| JP sales | sales in Japan, millions of copies |
| Other sales | sales in other countries, millions of copies |
| Critic Score | critic score (0-100) |
| User Score | user score (0-10) |
| Rating | ESRB age rating |

### Table of contents

1. Loading the data and taking a first look at it
2. Checking the data for errors and preprocessing it
   1. Column names
   2. Data types
   3. Missing values
   4. Explicit and implicit duplicates
3. Filtering the data (2000-2013)
4. Categorizing the data
5. Final conclusion
---

## Loading the data and taking a first look at it

- Load the required Python libraries and the `/datasets/new_games.csv` dataset.


In [56]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [57]:
# Importing the libraries
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)


In [58]:
# Loading the data, allowing for a possible difference in the path (local work / platform)
try:
    df = pd.read_csv('../datasets/new_games.csv')
except FileNotFoundError:
    print('File not found in the specified path. Please check the file path and try again.')

df.head(10)


,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.0,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN
6,New Super Mario Bros.,DS,2006.0,Platform,11.28,9.14,6.5,2.88,89.0,8.5,E
7,Wii Play,Wii,2006.0,Misc,13.96,9.18,2.93,2.84,58.0,6.6,E
8,New Super Mario Bros. Wii,Wii,2009.0,Platform,14.44,6.94,4.7,2.24,87.0,8.4,E
9,Duck Hunt,NES,1984.0,Shooter,26.93,0.63,0.28,0.47,NaN,NaN,NaN


In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  str    
 1   Platform         16956 non-null  str    
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  str    
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  str    
 6   JP sales         16956 non-null  str    
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  str    
 10  Rating           10085 non-null  str    
dtypes: float64(4), str(7)
memory usage: 1.4 MB


In [60]:
print('Table size (rows, columns):', df.shape)
print()
print('Column names:')
print(list(df.columns))


Table size (rows, columns): (16956, 11)

Column names:
['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales', 'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating']


#### Interim conclusion

The dataset has 16,956 rows and 11 columns. The column names and their contents generally match the data description.

The first look revealed the following:

- the `Name` and `Genre` columns each have 2 missing values;
- `Year of Release` is missing for 275 rows;
- the largest number of missing values is in `Critic Score`, `User Score` and `Rating`;
- `Year of Release` has the `float64` type, although a year is more logically stored as an integer;
- `EU sales`, `JP sales` and `User Score` have the `object` type, although they should contain numeric values. These columns most likely contain string values;
- the column names contain spaces and capital letters, so they need to be converted to snake_case.

During preprocessing we need to fix the column names, check the string values in the numeric columns, handle the missing values and examine the data for explicit and implicit duplicates.

---

##  Checking the data for errors and preprocessing it


### Data frame column names, or labels

In [61]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='str')

In [62]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [63]:
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='str')

The column names were converted to **snake_case**: every name is written in lower case and spaces were replaced with the `_` character. The column names now have a single format that is convenient to work with.


### Data types

In [64]:
df.dtypes

name                   str
platform               str
year_of_release    float64
genre                  str
na_sales           float64
eu_sales               str
jp_sales               str
other_sales        float64
critic_score       float64
user_score             str
rating                 str
dtype: object

In [65]:
print('EU sales:')
print(df['eu_sales'].unique())

print('\nJP sales:')
print(df['jp_sales'].unique())

print('\nUser score:')
print(df['user_score'].unique())

EU sales:
<StringArray>
['28.96',  '3.58', '12.76', '10.93',  '8.89',  '2.26',  '9.14',  '9.18',
  '6.94',  '0.63',
 ...
  '0.06',  '0.03',  '0.37',  '0.05',  '0.23',  '0.65',  '0.42',  '0.34',
  '0.35',  '0.78']
Length: 308, dtype: str

JP sales:
<StringArray>
[ '3.77',  '6.81',  '3.79',  '3.28', '10.22',  '4.22',   '6.5',  '2.93',
   '4.7',  '0.28',
 ...
  '1.26',  '0.85',  '0.43',  '0.67',  '1.14',  '0.86',  '1.17',   '0.5',
  '1.02',  '0.97']
Length: 245, dtype: str

User score:
<StringArray>
[  '8',   nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4', '8.2',
   '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2', '8.9', '6.4',
 '7.8', '7.5', '2.6', '7.2', '9.2',   '7', '7.3', '4.3', '7.6', '5.7',   '5',
 '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8', '6.1', '6.7', '5.4',   '4',
 '4.9', '4.5', '9.3', '6.2', '4.2',   '6', '3.7', '4.1', '5.8', '5.6', '5.5',
 '4.4', '4.6', '5.9', '3.9', '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5',
 '2.5', '1.9',   '3', '2.7', '

In [66]:
df[['eu_sales', 'jp_sales', 'user_score']].head(10)

,eu_sales,jp_sales,user_score
0,28.96,3.77,8
1,3.58,6.81,NaN
2,12.76,3.79,8.3
3,10.93,3.28,8
4,8.89,10.22,NaN
5,2.26,4.22,NaN
6,9.14,6.5,8.5
7,9.18,2.93,6.6
8,6.94,4.7,8.4
9,0.63,0.28,NaN


Checking the unique values showed that:

- `eu_sales` contains the string value `unknown`;
- `jp_sales` contains the string value `unknown`;
- `user_score` contains the string value `tbd`, as well as missing `NaN` values.

`TBD` stands for *to be determined* — the score has not been set yet. These values cannot be correctly interpreted as numbers, so they will be replaced with missing values during the conversion.


In [67]:
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

In [68]:
df.dtypes

name                   str
platform               str
year_of_release    float64
genre                  str
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating                 str
dtype: object

In [69]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  str    
 1   platform         16956 non-null  str    
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  str    
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  str    
dtypes: float64(7), str(4)
memory usage: 1.4 MB


#### Interim conclusion on data types

While checking the data types, incorrect types were found in the `eu_sales`, `jp_sales` and `user_score` columns. The sales columns contained the string value `unknown`, and the user scores contained `tbd`. Because of this, pandas read these columns as `object`.

The columns were converted to a numeric type with `pd.to_numeric()` using the `errors='coerce'` parameter. String values that cannot be converted to numbers were replaced with missing `NaN` values.

The `year_of_release` column still has the `float64` type. Since it contains missing values, the release year will be converted to an integer type after the missing values are handled.


### Missing values in the data

In [70]:
df.isna().sum()

name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [71]:
(df.isna().mean() * 100).round(2)

name                0.01
platform            0.00
year_of_release     1.62
genre               0.01
na_sales            0.00
eu_sales            0.04
jp_sales            0.02
other_sales         0.00
critic_score       51.39
user_score         54.66
rating             40.52
dtype: float64

In [72]:
missing_data = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
})

missing_data.sort_values('missing_count', ascending=False)

,missing_count,missing_percent
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01
platform,0,0.00
na_sales,0,0.00


#### Interim conclusion on missing values

Missing values were found in several columns. The largest number of them is in `user_score` — 9,268 values (54.66%), `critic_score` — 8,714 values (51.39%) and `rating` — 6,871 values (40.52%).

The `year_of_release` column is missing 275 values (1.62%). `name` and `genre` contain only 2 missing values each (0.01%).

In `eu_sales` and `jp_sales`, 6 and 4 missing values respectively appeared after the string value `unknown` was converted to `NaN`. `na_sales` and `other_sales` have no missing values.

The large number of missing values in the user and critic scores may be explained by the fact that some games were never scored or that the data about the scores was not available. Missing values in the ESRB rating can occur if a game was not assigned a rating or if information about it was not collected. Missing values in the release year cannot be reliably restored without additional information.


### Examining the rows with missing values

In [73]:
df[df['name'].isna() | df['genre'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


In [74]:
df[df['year_of_release'].isna()].head(10)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
183,Madden NFL 2004,PS2,NaN,Sports,4.26,0.26,0.01,0.71,94.0,8.5,E
379,FIFA Soccer 2004,PS2,NaN,Sports,0.59,2.36,0.04,0.51,84.0,6.4,E
458,LEGO Batman: The Videogame,Wii,NaN,Action,1.80,0.97,0.00,0.29,74.0,7.9,E10+
477,wwe Smackdown vs. Raw 2006,PS2,NaN,Fighting,1.57,1.02,0.00,0.41,NaN,NaN,NaN
611,Space Invaders,2600,NaN,Shooter,2.36,0.14,0.00,0.03,NaN,NaN,NaN
629,Rock Band,X360,NaN,Misc,1.93,0.33,0.00,0.21,92.0,8.2,T
659,Frogger's Adventures: Temple of the Frog,GBA,NaN,Adventure,2.15,0.18,0.00,0.07,73.0,NaN,E
680,LEGO Indiana Jones: The Original Adventures,Wii,NaN,Action,1.51,0.61,0.00,0.21,78.0,6.6,E10+
722,Call of Duty 3,Wii,NaN,Shooter,1.17,0.84,0.00,0.23,69.0,6.7,T
808,Rock Band,Wii,NaN,Misc,1.33,0.56,0.00,0.20,80.0,6.3,T


In [75]:
df[df['eu_sales'].isna() | df['jp_sales'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
446,Rhythm Heaven,DS,2008.0,Misc,0.55,NaN,1.93,0.13,83.0,9.0,E
467,Saints Row 2,X360,2008.0,Action,1.94,0.79,NaN,0.28,81.0,8.1,M
802,Dead Rising,X360,2006.0,Action,1.16,NaN,0.08,0.20,85.0,7.6,M
819,UFC 2009 Undisputed,X360,2009.0,Fighting,1.48,0.39,NaN,0.19,83.0,7.9,T
1131,Prince of Persia: Warrior Within,PS2,2004.0,Action,0.54,NaN,0.00,0.22,83.0,8.5,M
1132,Far Cry 4,XOne,2014.0,Shooter,0.80,NaN,0.01,0.14,82.0,7.5,M
1379,Hello Kitty Party,DS,2007.0,Misc,0.78,0.51,NaN,0.12,NaN,NaN,E
1394,Sonic Advance 3,GBA,2004.0,Platform,0.74,NaN,0.08,0.06,79.0,8.4,E
1612,Ratatouille,DS,2007.0,Action,0.49,NaN,0.00,0.14,NaN,NaN,NaN
4732,Castlevania: The Dracula X Chronicles,PSP,2007.0,Platform,0.22,0.09,NaN,0.07,80.0,7.8,T


#### Conclusion after examining the rows with missing values

The rows without `name` and `genre` are two records from 1993. Since the title and the genre cannot be reliably restored, and the rows themselves do not belong to the 2000-2013 period under review, they can be removed.

`year_of_release` is missing for 275 rows. Since the release year is required for the subsequent filtering by period, such records cannot be correctly assigned to the interval under review. It therefore makes sense to remove them as well.

`eu_sales` and `jp_sales` have a small number of missing values. According to the project requirements, such values can be replaced with the average sales volume for the corresponding platform and release year.

Missing values in `critic_score` and `user_score` are better left unfilled, so as not to create artificial scores. Missing values in `rating` can be replaced with a dedicated indicator for missing data.


### Handling the missing values

In [76]:
# Saving the number of rows before the removal
rows_before_missing = len(df)
rows_before_missing

16956

In [77]:
# Removing the rows without a title, a genre and a release year
df = df.dropna(
    subset=['name', 'genre', 'year_of_release']
).copy()

# Now the year can be converted to an integer type
df['year_of_release'] = df['year_of_release'].astype(int)

df[['name', 'genre', 'year_of_release']].isna().sum()

name               0
genre              0
year_of_release    0
dtype: int64

In [78]:
# Filling the missing values in the regional sales with the average
# for the corresponding platform and release year
sales_columns = ['eu_sales', 'jp_sales']

for column in sales_columns:
    df[column] = df[column].fillna(
        df.groupby(['platform', 'year_of_release'])[column]
          .transform('mean')
    )

df[['eu_sales', 'jp_sales']].isna().sum()

eu_sales    0
jp_sales    0
dtype: int64

In [79]:
# Checking that the indicator value UNKNOWN is not used yet
'UNKNOWN' in df['rating'].dropna().unique()

False

In [80]:
# Filling the missing ESRB rating with an indicator value
df['rating'] = df['rating'].fillna('UNKNOWN')

In [81]:
# Final check for missing values
df.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8594
user_score         9121
rating                0
dtype: int64

In [82]:
# Checking the data types after the processing
df.dtypes

name                   str
platform               str
year_of_release      int64
genre                  str
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating                 str
dtype: object

In [83]:
# Counting the number of removed rows
removed_rows = rows_before_missing - len(df)
removed_percent = removed_rows / rows_before_missing * 100

print('Rows removed:', removed_rows)
print(f'Rows removed: {removed_percent:.2f}%')

Rows removed: 277
Rows removed: 1.63%


### Explicit and implicit duplicates in the data

In [84]:
print('Platforms:')
print(sorted(df['platform'].unique()))

print('\nGenres:')
print(sorted(df['genre'].unique()))

print('\nESRB ratings:')
print(sorted(df['rating'].unique()))

print('\nNumber of unique game titles:')
print(df['name'].nunique())

Platforms:
['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG', 'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP', 'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360', 'XB', 'XOne']

Genres:
['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING', 'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform', 'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing', 'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter', 'Simulation', 'Sports', 'Strategy']

ESRB ratings:
['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN']

Number of unique game titles:
11426


In [85]:
df['name'] = df['name'].str.lower().str.strip()
df['genre'] = df['genre'].str.lower().str.strip()
df['rating'] = df['rating'].str.upper().str.strip()

In [86]:
print('Genres after normalization:')
print(sorted(df['genre'].unique()))

print('\nESRB ratings after normalization:')
print(sorted(df['rating'].unique()))

print('\nNumber of unique game titles after normalization:')
print(df['name'].nunique())

Genres after normalization:
['action', 'adventure', 'fighting', 'misc', 'platform', 'puzzle', 'racing', 'role-playing', 'shooter', 'simulation', 'sports', 'strategy']

ESRB ratings after normalization:
['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN']

Number of unique game titles after normalization:
11426


In [87]:
print('Number of explicit duplicates:')
print(df.duplicated().sum())

Number of explicit duplicates:
235


In [88]:
df[df.duplicated(keep=False)].sort_values(
    by=['name', 'platform', 'year_of_release']
).head(20)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
15301,11eyes: crossover,X360,2009,adventure,0.00,0.00,0.02,0.00,NaN,NaN,UNKNOWN
15302,11eyes: crossover,X360,2009,adventure,0.00,0.00,0.02,0.00,NaN,NaN,UNKNOWN
4860,18 wheeler: american pro trucker,PS2,2001,racing,0.20,0.15,0.00,0.05,61.0,5.7,E
4861,18 wheeler: american pro trucker,PS2,2001,racing,0.20,0.15,0.00,0.05,61.0,5.7,E
13098,4 elements,PC,2009,puzzle,0.00,0.04,0.00,0.01,NaN,7.4,E
13099,4 elements,PC,2009,puzzle,0.00,0.04,0.00,0.01,NaN,7.4,E
5235,"999: nine hours, nine persons, nine doors",DS,2009,adventure,0.31,0.00,0.03,0.02,NaN,NaN,UNKNOWN
5236,"999: nine hours, nine persons, nine doors",DS,2009,adventure,0.31,0.00,0.03,0.02,NaN,NaN,UNKNOWN
4958,alpha protocol,X360,2010,role-playing,0.23,0.12,0.00,0.04,63.0,7.2,M
4959,alpha protocol,X360,2010,role-playing,0.23,0.12,0.00,0.04,63.0,7.2,M


In [89]:
rows_before_duplicates = len(df)

df = df.drop_duplicates().copy()

removed_duplicates = rows_before_duplicates - len(df)

print('Explicit duplicates removed:', removed_duplicates)
print('Rows left:', len(df))

Explicit duplicates removed: 235
Rows left: 16444


In [90]:
print('Explicit duplicates after the removal:')
print(df.duplicated().sum())

Explicit duplicates after the removal:
0


#### Interim conclusion on duplicates

While examining the categorical data, implicit duplicates were found in the `genre` column, caused by values being written in different letter cases. To eliminate this problem, the game titles and genres were converted to lower case and the ESRB rating values to upper case. Extra spaces at the beginning and the end of the strings were removed as well.

After normalization, 235 explicit duplicates — completely identical rows — were found. They were removed with the `drop_duplicates()` method.

A repeat check after the removal showed that there are no explicit duplicates left in the data.

In [91]:
total_removed_rows = rows_before_missing - len(df)
total_removed_percent = total_removed_rows / rows_before_missing * 100

print('Original number of rows:', rows_before_missing)
print('Number of rows after preprocessing:', len(df))
print('Total rows removed:', total_removed_rows)
print(f'Share of removed rows: {total_removed_percent:.2f}%')

Original number of rows: 16956
Number of rows after preprocessing: 16444
Total rows removed: 512
Share of removed rows: 3.02%


### Overall interim conclusion on data preprocessing

The following was done during data preprocessing:

- the column names were converted to `snake_case`;
- the `eu_sales`, `jp_sales` and `user_score` columns were converted to a numeric type;
- the string values `unknown` and `tbd` were converted to missing values;
- rows without a game title, genre and release year were removed;
- the `year_of_release` column was converted to an integer type;
- missing values in `eu_sales` and `jp_sales` were filled with the average values for the corresponding platform and release year;
- missing values in `critic_score` and `user_score` were left unfilled, so as not to create artificial scores;
- missing values in `rating` were replaced with the indicator value `UNKNOWN`;
- the text values were normalized by letter case;
- implicit duplicates in the genre names were eliminated;
- 235 explicit duplicates were found and removed.

In total, 512 rows were removed during preprocessing, which is 3.02% of the original amount of data. After preprocessing, 16,444 rows remained in the data frame.

---

##  Filtering the data


In [92]:
df_actual = df[
    (df['year_of_release'] >= 2000) &
    (df['year_of_release'] <= 2013)
].copy()

print('Number of rows in df_actual:', len(df_actual))
print('Minimum year:', df_actual['year_of_release'].min())
print('Maximum year:', df_actual['year_of_release'].max())

Number of rows in df_actual: 12781
Minimum year: 2000
Maximum year: 2013


In [93]:
df_actual['year_of_release'].value_counts().sort_index()

year_of_release
2000     350
2001     482
2002     829
2003     775
2004     762
2005     939
2006    1006
2007    1197
2008    1427
2009    1426
2010    1255
2011    1136
2012     653
2013     544
Name: count, dtype: int64

#### Interim conclusion on data filtering

A `df_actual` data frame containing only the games released between 2000 and 2013 inclusive was created for further analysis.

After filtering, 12,781 rows remained in the data frame. Checking the minimum and the maximum year returned 2000 and 2013 respectively, so the selected time interval matches the project requirements.

---

## Categorizing the data


In [94]:
def categorize_user_score(score):
    if pd.isna(score):
        return np.nan
    elif score >= 8:
        return 'high score'
    elif score >= 3:
        return 'medium score'
    else:
        return 'low score'

In [95]:
df_actual['user_score_category'] = (
    df_actual['user_score'].apply(categorize_user_score)
)

df_actual[
    ['user_score', 'user_score_category']
].head(15)

,user_score,user_score_category
0,8.0,high score
2,8.3,high score
3,8.0,high score
6,8.5,high score
7,6.6,medium score
8,8.4,high score
10,NaN,NaN
11,8.6,high score
13,7.7,medium score
14,6.3,medium score


In [96]:
df_actual['user_score_category'].value_counts(dropna=False)

user_score_category
NaN             6298
medium score    4081
high score      2286
low score        116
Name: count, dtype: int64

In [97]:
df_actual[
    df_actual['user_score'].isin([0, 3, 8, 10])
][['user_score', 'user_score_category']].drop_duplicates().sort_values('user_score')

,user_score,user_score_category
2864,0.0,low score
2694,3.0,medium score
0,8.0,high score


In [98]:
def categorize_critic_score(score):
    if pd.isna(score):
        return np.nan
    elif score >= 80:
        return 'high score'
    elif score >= 30:
        return 'medium score'
    else:
        return 'low score'

In [99]:
df_actual['critic_score_category'] = (
    df_actual['critic_score'].apply(categorize_critic_score)
)

df_actual[
    ['critic_score', 'critic_score_category']
].head(15)

,critic_score,critic_score_category
0,76.0,medium score
2,82.0,high score
3,80.0,high score
6,89.0,high score
7,58.0,medium score
8,87.0,high score
10,NaN,NaN
11,91.0,high score
13,80.0,high score
14,61.0,medium score


In [100]:
df_actual['critic_score_category'].value_counts(dropna=False)

critic_score_category
NaN             5612
medium score    5422
high score      1692
low score         55
Name: count, dtype: int64

In [101]:
df_actual[
    df_actual['critic_score'].isin([0, 30, 80, 100])
][['critic_score', 'critic_score_category']].drop_duplicates().sort_values('critic_score')

,critic_score,critic_score_category
1576,30.0,medium score
3,80.0,high score


#### Interim conclusion on categorization

The games were split into categories by user and critic scores.

By user score:
- high score — 2,286 games;
- medium score — 4,081 games;
- low score — 116 games;
- 6,298 games have no user score.

By critic score:
- high score — 1,692 games;
- medium score — 5,422 games;
- low score — 55 games;
- 5,612 games have no critic score.

Checking the boundary values showed that the categories were formed correctly: the value 3 falls into the medium user score, 8 — into the high one; the value 30 falls into the medium critic score, and 80 — into the high one.

In [102]:
top_7_platforms = df_actual['platform'].value_counts().head(7)

top_7_platforms

platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64

In [103]:
print('Top 7 platforms by the number of games released:')
print(top_7_platforms)

Top 7 platforms by the number of games released:
platform
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64


---

## Final conclusion


As part of this project, historical video game data was preprocessed and a data slice was prepared for analysing the development of the gaming industry between 2000 and 2013 inclusive.

The work started with a review of the original dataset, which contained 16,956 rows and 11 columns. Missing values, incorrect data types, differences in the spelling of categorical values and duplicates were found in it.

During preprocessing:

- the column names were converted to `snake_case`;
- the `eu_sales`, `jp_sales` and `user_score` columns were converted to a numeric type;
- the string values `unknown` and `tbd` were converted to missing values;
- rows without a game title, genre and release year were removed;
- `year_of_release` was converted to an integer type;
- missing values in the European and Japanese sales were filled with the average values for the corresponding platform and release year;
- missing values in the user and critic scores were left unfilled, so as not to create artificial data;
- missing ESRB rating values were replaced with the `UNKNOWN` indicator;
- the text values were normalized by letter case;
- implicit duplicates in the genre names were eliminated;
- 235 explicit duplicates were found and removed.

In total, 512 rows were removed during preprocessing, which is 3.02% of the original dataset. After preprocessing, 16,444 rows remained.

A `df_actual` data frame that includes only the games released between 2000 and 2013 inclusive was created for further research. The final slice contains 12,781 records.

Two new fields were also added:

- `user_score_category` — the user score category;
- `critic_score_category` — the critic score category.

The games were split into categories with a high, medium and low score. Missing scores were kept as missing values during categorization.

By user score:
- high score — 2,286 games;
- medium score — 4,081 games;
- low score — 116 games;
- 6,298 games have no score.

By critic score:
- high score — 1,692 games;
- medium score — 5,422 games;
- low score — 55 games;
- 5,612 games have no score.

Top 7 platforms by the number of games released in 2000-2013:

1. PS2 — 2,127 games;
2. DS — 2,120 games;
3. Wii — 1,275 games;
4. PSP — 1,180 games;
5. X360 — 1,121 games;
6. PS3 — 1,087 games;
7. GBA — 811 games.

The data has thus been cleaned and prepared for further research into the gaming industry of the early 21st century. The resulting slice makes it possible to analyse the platforms, genres, sales and scores of the games for the required period.

In [104]:
print('Final slice size:', df_actual.shape)
print('Period:', df_actual['year_of_release'].min(), '-', df_actual['year_of_release'].max())
print('Explicit duplicates:', df_actual.duplicated().sum())
print('\nMissing values in the new categories:')
print(df_actual[['user_score_category', 'critic_score_category']].isna().sum())

Final slice size: (12781, 13)
Period: 2000 - 2013
Explicit duplicates: 0

Missing values in the new categories:
user_score_category      6298
critic_score_category    5612
dtype: int64
